In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from IPython.display import display, clear_output
import ipywidgets as widgets

class DonnanSimulation:
    def __init__(self):
        # System constants
        self.R = 8.314
        self.T = 310.15
        self.F = 96485
        self.P_w = 1e-9
        self.A = 1e-4
        
        # Initial volumes
        self.V_i = 1.0e-6
        self.V_c = 1.2e-6
        self.total_volume = self.V_i + self.V_c
        
        # Initial ion concentrations (mol/m³)
        self.Na_c = 145
        self.Na_i = 140
        self.Cl_c = 110
        self.Cl_i = 108

    def calculate_osmotic_pressure(self, c1, c2):
        return self.R * self.T * (c1 - c2)

    def calculate_membrane_potential(self, Na_i, Na_c, Cl_i, Cl_c):
        return (self.R * self.T / self.F) * np.log((Na_c + Cl_i) / (Na_i + Cl_c))

    def assess_edema_risk(self, volume_change, time_constant):
        if volume_change > 20:
            risk = "High"
            explanation = "Significant volume increase indicates high risk of edema formation"
            color = 'red'
        elif volume_change > 10:
            risk = "Moderate"
            explanation = "Volume increase suggests potential for edema development"
            color = 'orange'
        elif volume_change > 5:
            risk = "Low"
            explanation = "Minor volume changes indicate low risk of edema"
            color = 'yellow'
        else:
            risk = "Minimal"
            explanation = "Volume changes within normal physiological range"
            color = 'green'
            
        if time_constant < 600:
            speed = "rapid"
        else:
            speed = "gradual"
            
        return risk, explanation, color, speed

    def system_dynamics(self, t, state, P_c, P_i):
        V_i, Na_i, Cl_i = state
        
        V_c = self.total_volume - V_i
        
        if V_i <= 0.1e-6 or V_i >= 1.9e-6:
            return [0, 0, 0]
            
        Na_c = self.Na_c * self.V_c / V_c
        Cl_c = self.Cl_c * self.V_c / V_c
        
        pi_protein = self.calculate_osmotic_pressure(P_c, P_i)
        pi_ions = self.calculate_osmotic_pressure(Na_c + Cl_c, Na_i + Cl_i)
        
        J_v = self.P_w * self.A * (pi_protein + pi_ions) * (1 - abs(V_i - self.V_i) / self.V_i)
        
        dV_i_dt = J_v
        dNa_i_dt = -Na_i * J_v / V_i + 0.1 * (Na_c - Na_i)
        dCl_i_dt = -Cl_i * J_v / V_i + 0.1 * (Cl_c - Cl_i)
        
        return [dV_i_dt, dNa_i_dt, dCl_i_dt]

    def simulate(self, P_c, P_i=0.6, t_span=3600):
        y0 = [self.V_i, self.Na_i, self.Cl_i]
        t = np.linspace(0, t_span, 500)
        
        solution = solve_ivp(
            lambda t, y: self.system_dynamics(t, y, P_c, P_i),
            [0, t_span],
            y0,
            t_eval=t,
            method='RK45',
            rtol=1e-6,
            atol=1e-8
        )
        
        return solution.t, solution.y[0], solution.y[1], solution.y[2]

def create_interactive_simulation():
    sim = DonnanSimulation()
    
    # Create widgets
    protein_slider = widgets.FloatSlider(
        value=1.0,
        min=0.5,
        max=2.0,
        step=0.1,
        description='Capillary Protein (mol/m³):',
        style={'description_width': 'initial'},
        layout={'width': '50%'}
    )
    
    output_plots = widgets.Output()
    output_text = widgets.Output()
    
    def update(change):
        P_c = change['new']
        P_i = 0.6  # Fixed interstitial protein
        
        # Run simulation
        t, V_i, Na_i, Cl_i = sim.simulate(P_c, P_i)
        
        # Calculate metrics
        volume_change_percent = ((V_i[-1] - V_i[0]) / V_i[0]) * 100
        target_volume = V_i[0] + 0.63 * (V_i[-1] - V_i[0])
        time_constant = t[np.argmin(np.abs(V_i - target_volume))]
        
        # Get risk assessment
        risk, explanation, color, speed = sim.assess_edema_risk(volume_change_percent, time_constant)
        
        # Update plots
        with output_plots:
            clear_output(wait=True)
            
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
            
            # Volume plot
            ax1.plot(t, V_i * 1e6, 'b-', label='Interstitial Volume')
            ax1.set_xlabel('Time (s)')
            ax1.set_ylabel('Volume (µL)')
            ax1.set_title(f'Interstitial Volume Changes\nVolume Change: {volume_change_percent:.1f}%')
            ax1.grid(True)
            ax1.legend()
            
            # Ion concentrations plot
            ax2.plot(t, Na_i, 'r-', label='Na⁺')
            ax2.plot(t, Cl_i, 'g-', label='Cl⁻')
            ax2.set_xlabel('Time (s)')
            ax2.set_ylabel('Concentration (mol/m³)')
            ax2.set_title('Ion Concentrations')
            ax2.grid(True)
            ax2.legend()
            
            plt.tight_layout()
            display(fig)
            plt.close()
        
        # Update text output
        with output_text:
            clear_output(wait=True)
            print(f"\nAnalysis at {P_c:.1f} mol/m³ capillary protein concentration:")
            print(f"----------------------------------------")
            print(f"Volume Change: {volume_change_percent:.1f}%")
            print(f"Time Constant: {time_constant:.1f} seconds")
            print(f"Edema Risk: {risk} ({color})")
            print(f"Assessment: {explanation}")
            print(f"Rate of Change: Changes occur at a {speed} rate")
            if volume_change_percent > 10:
                print("\nRecommendation: Monitor closely for edema development")
            if speed == "rapid" and volume_change_percent > 5:
                print("Warning: Rapid volume changes may require immediate attention")
    
    # Register callback
    protein_slider.observe(update, names='value')
    
    # Initial update
    update({'new': protein_slider.value})
    
    # Layout
    display(widgets.VBox([
        protein_slider,
        output_plots,
        output_text
    ]))

# Create and display the interactive simulation
create_interactive_simulation()